# Module 3 — TF-IDF-Weighted Word2Vec

Train Skip-Gram Word2Vec, create TF-IDF-weighted document vectors, and save the Logistic Regression component used by the ensemble.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_DIR, RANDOM_SEED
from src.dataset_utils import load_fixed_data_splits
from src.evaluation import calculate_metrics, save_evaluation_outputs, update_metrics_file
from src.word_embeddings import tokenize, weighted_document_vectors

train_data, validation_data, test_data = load_fixed_data_splits()

## Train Skip-Gram Word2Vec and TF-IDF

In [ ]:
training_tokens = train_data['processed_text'].map(tokenize).tolist()
word2vec = Word2Vec(
    sentences=training_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=1,
    epochs=10,
    seed=RANDOM_SEED,
)
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r'(?u)\S+',
    min_df=2,
)
tfidf_vectorizer.fit(train_data['processed_text'])

## Build weighted document vectors and train Logistic Regression

In [ ]:
train_features = weighted_document_vectors(train_data['processed_text'], tfidf_vectorizer, word2vec)
validation_features = weighted_document_vectors(validation_data['processed_text'], tfidf_vectorizer, word2vec)
test_features = weighted_document_vectors(test_data['processed_text'], tfidf_vectorizer, word2vec)

classifier = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=RANDOM_SEED,
)
classifier.fit(train_features, train_data['label'])

## Evaluate and save the ensemble component

In [ ]:
validation_predictions = classifier.predict(validation_features)
validation_f1 = calculate_metrics(validation_data['label'], validation_predictions)['Macro F1']
test_predictions = classifier.predict(test_features)
result = save_evaluation_outputs(
    experiment_id='M3.2',
    experiment_name='TF-IDF Weighted Word2Vec + Logistic Regression',
    family='Dense Embedding',
    test_data=test_data,
    predictions=test_predictions.tolist(),
)
result['Representation'] = 'TF-IDF Weighted Word2Vec'
result['Validation Macro F1'] = validation_f1
selected_metrics = update_metrics_file(pd.DataFrame([result]))

joblib.dump({
    'name': 'TF-IDF Weighted Word2Vec + Logistic Regression',
    'word2vec': word2vec,
    'tfidf_vectorizer': tfidf_vectorizer,
    'classifier': classifier,
    'classes': classifier.classes_.tolist(),
    'representation': 'weighted',
    'vector_size': 100,
    'validation_macro_f1': validation_f1,
}, MODEL_DIR / 'word2vec_sentiment.pkl')
selected_metrics[selected_metrics['Experiment ID'] == 'M3.2']